not investment advice

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

In [3]:
np.random.seed(42)

In [5]:
sp500 = list(pd.read_csv('sp500_companies.csv')['Symbol'])

In [6]:
for tick in ['ANSS', 'DFS', 'JNPR', 'WBA', 'HES', 'PARA', 'GEV', 'SOLV', 'AMTM']:
    sp500.remove(tick)

In [9]:
# ticker = sp500[np.random.randint(len(sp500))]
ticker = 'BA'
stock = yf.Ticker(ticker)

In [10]:
ticker

'BA'

In [11]:
# assert 'freeCashflow' in stock.info.keys()
# assert 'beta' in stock.info.keys()
# assert 'revenueGrowth' in stock.info.keys()
# assert 'marketCap' in stock.info.keys()

### Intrinsic Value
Intrinsic value aims to estimate what a firm is fundamentally worth based on its ability to generate future cash flows, adjusted for risk. However, valuation becomes most actionable when intrinsic value is compared directly to market prices. This comparison produces a valuation gap, a quantitative measure of how far market perception deviates from fundamental value.

In [2]:
def intrinsic_value(fcf, beta, growth, rf=0.05, equity_risk_premium=0.06, max_growth=0.05):
    g = min(growth, max_growth)
    r = rf + beta * equity_risk_premium
    
    if r <= g:
        return np.nan
    
    return fcf*(1+g)/(r-g)

In [13]:
fcf = stock.info['freeCashflow']
beta = stock.info['beta']
growth = stock.info['revenueGrowth']
market_cap = stock.info['marketCap']
intrinsic = intrinsic_value(fcf, beta, growth)
valuation_gap = np.log(market_cap / intrinsic)

C:\Users\Cassidy\AppData\Local\Temp\ipykernel_6272\3953821374.py:6: RuntimeWarning: invalid value encountered in log
  valuation_gap = np.log(market_cap / intrinsic)


In [14]:
print(f"Intrinsic Value: ${intrinsic:,.0f}")
print(f"Market Cap: ${market_cap:,.0f}")
print(f"Log Valuation Gap: {valuation_gap:.3f}")

Intrinsic Value: $-71,668,366,638
Market Cap: $178,634,211,328
Log Valuation Gap: nan


In [64]:
top10 = sp500[:10]
random10 = np.random.choice(sp500, 10, replace=False).tolist()

In [65]:
log_gaps = {}
try:
    for stock in random10:
        ticker = stock
        stock = yf.Ticker(ticker)
        fcf = stock.info['freeCashflow']
        beta = stock.info['beta']
        growth = stock.info['revenueGrowth']
        market_cap = stock.info['marketCap']
        intrinsic = intrinsic_value(fcf, beta, growth)
        valuation_gap = np.log(market_cap / intrinsic)
        print(f"{ticker}: Log Valuation Gap: {valuation_gap:.3f}")
        log_gaps[ticker] = valuation_gap
except KeyError:
    print(f"Data missing for {ticker}, skipping...")

ABNB: Log Valuation Gap: 0.476
BDX: Log Valuation Gap: -1.409
UBER: Log Valuation Gap: 0.550
MU: Log Valuation Gap: 4.258
HLT: Log Valuation Gap: 1.001
Data missing for J, skipping...


In [66]:
sorted(log_gaps.items(), key=lambda item: item[1])

[('BDX', np.float64(-1.4093324673882122)),
 ('ABNB', np.float64(0.47552292172796007)),
 ('UBER', np.float64(0.5504894597949466)),
 ('HLT', np.float64(1.0009890515326072)),
 ('MU', np.float64(4.258271493531301))]

### Price-to-Earnings
- How much the market is willing to pay for $1 of current earnings
- Low P/E -> market expects low growth, high risk, or temporary earnings
- High P/E -> market expects strong growth or stable earnings
- distorted by accounting choices
- meaningless for firms with negative earnings
- P/E = Price / EPS

### Other methods to research
- P/B
- EV/EBITDA
- Sharpe ratio

In [24]:
stock.info['enterpriseToEbitda']

-33.82

In [16]:
stock.info['previousClose'] / stock.info['trailingEps']

-16.776642335766425

In [17]:
stock.info['priceToBook']

-20.990063

In [67]:
#### not investment advice!!!